# Desalinhamento carpal em radiografia PA de punho: demonstração

Isabela de Avellar Meireles, Universidade Federal Fluminense (UFF)

Uma CNN localiza 11 pontos anatômicos do carpo e calcula medidas geométricas interpretáveis. Consulte o `README.md` para documentação, limitações e descrição dos dados.

Execute no Google Colab com GPU (Ambiente de execução > Alterar o tipo de ambiente > T4 GPU). Os dados anotados são necessários; veja a estrutura descrita abaixo.


## Dependências e dados

Fora do Colab, instale `tensorflow numpy pandas pillow opencv-python matplotlib scipy`.

Estrutura esperada em `RAIZ`: `para_anotar/` com imagens PNG e `chave.csv`, além de `pontos6_carpo.json` e `pontos_arco.json`. Os arquivos `pontos_casos_positivos.json`, `pontos11_repet_isa.json` e `chave_repet_isa.json` são opcionais.


In [ ]:
import sys, platform
EM_COLAB = 'google.colab' in sys.modules
if not EM_COLAB:
    %pip -q install tensorflow numpy pandas pillow opencv-python matplotlib scipy
print('Python', platform.python_version(), '| Colab:', EM_COLAB)


In [ ]:
import json
from pathlib import Path
import numpy as np, pandas as pd, cv2, matplotlib.pyplot as plt, tensorflow as tf
from PIL import Image
from scipy import stats
from tensorflow.keras import layers, Model

if EM_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    RAIZ = Path('/content/drive/MyDrive')
else:
    RAIZ = Path('dados')
IMGS, ANOT, SAIDA = RAIZ / 'para_anotar', RAIZ, RAIZ
LADO, HM, SIGMA, NP = 384, 144, 3.0, 11
SEMENTE, FRAC_TREINO, BATCH = 0, 0.8, 4
CONF_MIN, PIRES_FAIXA = 0.20, (1.46, 1.67)
RUINS = {'108.png', '136.png', '141.png', '147.png'}
tf.keras.utils.set_random_seed(SEMENTE)
GY, GX = np.mgrid[0:HM, 0:HM].astype(np.float32)


In [ ]:
def mapas(P):
    Q = P * (HM / LADO)
    return np.stack([np.exp(-((GX-x)**2+(GY-y)**2)/(2*SIGMA**2)) for x,y in Q], -1).astype(np.float32)

def decodifica(hm):
    pts, conf = [], []
    for m in np.moveaxis(hm, -1, 0):
        j = np.unravel_index(m.argmax(), m.shape); conf.append(float(m[j]))
        y0,y1=max(j[0]-2,0),min(j[0]+3,HM); x0,x1=max(j[1]-2,0),min(j[1]+3,HM)
        w=m[y0:y1,x0:x1]; gy,gx=np.mgrid[y0:y1,x0:x1]; s=w.sum()+1e-9
        pts.append([(w*gx).sum()/s,(w*gy).sum()/s])
    return np.array(pts),np.array(conf)

def circulo(Q):
    x,y=Q[:,0],Q[:,1]; A=np.column_stack([2*x,2*y,np.ones(len(x))])
    s,*_=np.linalg.lstsq(A,x*x+y*y,rcond=None)
    return np.array(s[:2]),float(np.sqrt(max(s[2]+s[0]**2+s[1]**2,1e-9)))

def medidas(P, ps=np.nan):
    P=np.asarray(P,float); esc_u,sem_r,cprox,cdist,mc3,radio=P[:6]
    v=cdist-cprox; L=float(np.hypot(*v))
    if L<5: return None
    u=v/L; cen,R=circulo(P[6:]); d=np.hypot(*(P[6:]-cen).T)
    alt=abs(float(np.dot(mc3-radio,u))); sl=float(np.hypot(*(esc_u-sem_r)))
    return {'pires':alt/L,'sl_sobre_capitato':sl/L,'residuo_arco':float(np.abs(d-R).max())/R,
            'degrau_sl':abs(float(d[1]-d[2]))/R,'degrau_lt':abs(float(d[3]-d[4]))/R,
            'raio_rel':R/L,'cap_mm':L*ps,'alt_mm':alt*ps,'sl_mm':sl*ps}


## Carregamento, divisão por paciente e treinamento

O conjunto de validação é separado por paciente, estratificado por origem. Este notebook treina um localizador de pontos por mapas de calor com MobileNetV2. Os resultados de validação não constituem teste independente; consulte as ressalvas detalhadas no README.


In [ ]:
p6=json.loads((ANOT/'pontos6_carpo.json').read_text()); arc=json.loads((ANOT/'pontos_arco.json').read_text())
chave=pd.read_csv(IMGS/'chave.csv').set_index('arquivo')
if 'origem' not in chave: chave['origem']=np.nan
chave['origem']=chave['origem'].fillna('GRAZ')
nomes=[n for n in sorted(p6) if len(p6[n])==6 and len(arc.get(n,[]))==5 and n not in RUINS and (IMGS/n).exists()]
orig=np.array([chave.loc[n,'origem'] if n in chave.index else 'GRAZ' for n in nomes])
pac=np.array([str(chave.loc[n,'patient_id']) if n in chave.index else n for n in nomes])
ps=np.array([float(chave.loc[n,'pixel_spacing']) if n in chave.index else np.nan for n in nomes])
X=np.zeros((len(nomes),LADO,LADO),np.uint8); PT=np.zeros((len(nomes),NP,2),np.float32); PO=np.zeros_like(PT); WH=np.zeros((len(nomes),2),np.float32)
for i,n in enumerate(nomes):
    im=Image.open(IMGS/n).convert('L'); W,H=im.size
    X[i]=np.asarray(im.resize((LADO,LADO)),np.uint8); PO[i]=np.array(p6[n]+arc[n],np.float32)
    PT[i]=PO[i]*[LADO/W,LADO/H]; WH[i]=W,H
itr=np.zeros(len(nomes),bool)
for o in np.unique(orig):
    u=np.unique(pac[orig==o]); np.random.default_rng(SEMENTE).shuffle(u)
    itr |= (orig==o)&np.isin(pac,list(set(u[:int(len(u)*FRAC_TREINO)])))
iva=np.where(~itr)[0]; print(len(nomes),'imagens; validação:',len(iva))

def monta_modelo():
    ent=layers.Input((LADO,LADO,1)); x=layers.Concatenate()([ent,ent,ent])
    base=tf.keras.applications.MobileNetV2(input_shape=(LADO,LADO,3),include_top=False,weights='imagenet')
    feat=Model(base.input,base.get_layer('block_13_expand_relu').output); feat.trainable=False; x=feat(x,training=False)
    for f in (128,64):
        x=layers.UpSampling2D(2)(x); x=layers.Conv2D(f,3,padding='same',activation='relu')(x); x=layers.BatchNormalization()(x)
    x=layers.Resizing(HM,HM)(x); return Model(ent,layers.Conv2D(NP,3,padding='same',activation='sigmoid')(x)),feat

Xva=(X[iva].astype(np.float32)/255)[...,None]; Yva=np.stack([mapas(PT[i]) for i in iva])
modelo,feat=monta_modelo(); print('parâmetros:',modelo.count_params())


In [ ]:
# Gerador com aumento geométrico e fotométrico sincronizado com os pontos.
class Gerador(tf.keras.utils.Sequence):
    def __init__(self, idx): super().__init__(); self.ordem=np.array(idx); self.rng=np.random.default_rng(SEMENTE)
    def __len__(self): return int(np.ceil(len(self.ordem)/BATCH))
    def __getitem__(self,b):
        xs,ys=[],[]
        for i in self.ordem[b*BATCH:(b+1)*BATCH]:
            img=X[i]; P=PT[i]
            for _ in range(10):
                M=cv2.getRotationMatrix2D((LADO/2,LADO/2),self.rng.uniform(-12,12),self.rng.uniform(.70,1.35))
                M[:,2]+=self.rng.uniform(-.08,.08,2)*LADO; Q=(M@np.hstack([P,np.ones((NP,1))]).T).T
                if Q.min()>8 and Q.max()<LADO-8: break
            else: Q=P; M=np.array([[1,0,0],[0,1,0]],float)
            a=cv2.warpAffine(img,M,(LADO,LADO)).astype(np.float32)
            a=np.clip(a*self.rng.uniform(.82,1.18)+self.rng.uniform(-18,18),0,255)
            a=255*(a/255)**self.rng.uniform(.80,1.25)
            xs.append(a/255); ys.append(mapas(Q))
        return np.stack(xs)[...,None],np.stack(ys)
    def on_epoch_end(self): self.rng.shuffle(self.ordem)

tr=Gerador(np.where(itr)[0])
class ErroPontos(tf.keras.callbacks.Callback):
    def on_epoch_end(self,epoch,logs=None):
        pred=self.model.predict(Xva,verbose=0)
        err=[np.hypot(*(decodifica(pred[j])[0]-PT[i]*HM/LADO).T) for j,i in enumerate(iva)]
        logs['val_erro_px']=float(np.median(err))

TREINAR=True
if TREINAR:
    cb=[ErroPontos(),tf.keras.callbacks.EarlyStopping(patience=25,restore_best_weights=True)]
    modelo.compile(tf.keras.optimizers.Adam(1e-3),loss='mse')
    h1=modelo.fit(tr,validation_data=(Xva,Yva),epochs=200,callbacks=cb,verbose=2)
    feat.trainable=True
    for c in feat.layers[:-30]: c.trainable=False
    modelo.compile(tf.keras.optimizers.Adam(1e-5),loss='mse')
    h2=modelo.fit(tr,validation_data=(Xva,Yva),epochs=80,callbacks=cb,verbose=2)
    modelo.save(SAIDA/'heatmap11_aug.keras')
else: modelo=tf.keras.models.load_model(SAIDA/'heatmap11_aug.keras',compile=False)


## Avaliação exploratória

Os pontos previstos são convertidos à resolução original e comparados às marcações manuais. As métricas apresentadas são exploratórias e a validação foi usada no early stopping. Não use como ferramenta diagnóstica; consulte as limitações e tabelas de resultados no README.


In [ ]:
Pva=modelo.predict(Xva,verbose=0); dec=[decodifica(h) for h in Pva]
E=np.array([np.hypot(*(p-PT[i]*HM/LADO).T) for (p,_),i in zip(dec,iva)])
CONF=np.array([c for _,c in dec])
print('Erro em pixels do mapa 144: mediana',round(float(np.median(E)),3),'p90',round(float(np.percentile(E,90)),3))
for o in np.unique(orig[iva]):
    m=orig[iva]==o; print(o,'n=',m.sum(),'mediana=',round(float(np.median(E[m])),3),'p90=',round(float(np.percentile(E[m],90)),3))
lin=[]
for j,i in enumerate(iva):
    pr=dec[j][0]*(WH[i]/HM); mp,mm=medidas(pr,ps[i]),medidas(PO[i],ps[i])
    if mp and mm: lin.append({'arquivo':nomes[i],**{k+'_p':mp[k] for k in ('pires','sl_sobre_capitato','residuo_arco','degrau_sl','degrau_lt')},**{k+'_m':mm[k] for k in ('pires','sl_sobre_capitato','residuo_arco','degrau_sl','degrau_lt')}})
VAL=pd.DataFrame(lin); print('Medidas modelo (p) e manuais (m):',len(VAL),'imagens')
for k in ('pires','sl_sobre_capitato','residuo_arco','degrau_sl','degrau_lt'):
    d=VAL[k+'_p']-VAL[k+'_m']; print(k,'erro absoluto mediano=',round(float(np.median(np.abs(d))),4),'viés=',round(float(d.mean()),4))
